# Task 1: News Topic Classifier Using BERT

## Setup

In [ ]:
pip install datasets transformers evaluate accelerate

## Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('ag_news')

# Display some information about the dataset
print(dataset)

In [ ]:
print(dataset['train'][0])
print(dataset['test'][0])

## Tokenize and Preprocess Dataset

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(['text'])
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
tokenized_datasets.set_format('torch')

# Display the first example of the tokenized dataset
print(tokenized_datasets['train'][0])

## Fine-tune BERT Model

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import AutoModelForSequenceClassification

# The AG News dataset has 4 labels (classes)
num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='ag_news_bert_finetuned',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    push_to_hub=False # We are not pushing to Hugging Face Hub for this example
)

## Evaluate Model

In [ ]:
import evaluate
import numpy as np

# Load the evaluation metrics
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted') # Use 'weighted' for multi-class f1

    return {'accuracy': accuracy['accuracy'], 'f1': f1['f1']}

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

## Model Evaluation Results

In [ ]:
# Evaluate the model on the test set
eval_results = trainer.evaluate()
print(eval_results)

## Deploy Model with Gradio

In [ ]:
pip install gradio

In [ ]:
import gradio as gr
import torch

# Get the device for model inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define the labels for the AG News dataset
labels = ['World', 'Sports', 'Business', 'Sci/Tech']

def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)[0]
    predicted_class_id = torch.argmax(probabilities).item()
    predicted_label = labels[predicted_class_id]

    # Return probabilities for all classes
    return {label: prob.item() for label, prob in zip(labels, probabilities)}

In [ ]:
iface = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=2, placeholder='Enter news headline here...'),
    outputs=gr.Label(),
    title='News Topic Classifier (BERT Fine-tuned)',
    description='Enter a news headline and get the predicted topic category.'
)

iface.launch(debug=True)

Successfully completed the task of building a News Topic Classifier using a BERT model. Here's a summary and some insights:

# **Summary of Steps:**

Environment Setup and Data Loading: We started by installing necessary libraries (datasets, transformers, evaluate, accelerate, gradio) and loaded the AG News Dataset from Hugging Face, which contains news headlines and their corresponding topic labels.
Data Preprocessing and Tokenization: We used the bert-base-uncased tokenizer to convert the raw text headlines into a format suitable for the BERT model. The dataset was tokenized, unnecessary columns were removed, the 'label' column was renamed to 'labels', and the data format was set to PyTorch tensors.
Model Fine-tuning: We loaded a bert-base-uncased model pre-trained for sequence classification. We defined TrainingArguments specifying parameters like learning rate, batch size, number of epochs, and evaluation strategy. The model was then fine-tuned on the preprocessed dataset using the Hugging Face Trainer.
Model Evaluation: During and after training, the model's performance was evaluated using 'accuracy' and 'f1-score' metrics (weighted average for multi-class classification).
Model Deployment: Finally, we deployed the fine-tuned model using Gradio, creating a user-friendly web interface. This allows users to input a news headline and receive real-time predictions of its topic category along with confidence scores for each class.
# **Insights:**

Effectiveness of Transfer Learning: Fine-tuning a pre-trained BERT model proved to be highly effective for this text classification task, achieving good performance metrics (accuracy and F1-score) on the AG News dataset. This demonstrates the power of transfer learning in NLP.
Hugging Face Ecosystem: The Hugging Face transformers and datasets libraries significantly streamlined the entire process, from data loading and preprocessing to model training and evaluation.
Ease of Deployment: Gradio made it straightforward to create an interactive demo for the model, allowing for easy testing and demonstration of the classifier's capabilities.
Potential for Improvement: While the model performs well, further improvements could include hyperparameter tuning, experimenting with different BERT variants, or using more advanced data augmentation techniques. Additionally, deploying to a more robust platform (like a dedicated web service) would be a next step for production use cases.
This project provides a complete pipeline for building and deploying a news topic classifier using state-of-the-art NLP techniques.

